In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time 
from datetime import datetime
import json

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

Função recursiva 

In [2]:
def buscar_ofertas_completas(dado, lista_ofertas):
    if isinstance(dado, dict):
        if ('sellerName' in dado or 'merchantName' in dado) and 'price' in dado:
            oferta = {
                'loja': dado.get('sellerName', dado.get('merchantName')),
                'preco_vista': dado.get('price'),
                'nome_anuncio': dado.get('name'),
                'id_oferta': dado.get('id'),
                'preco_total': dado.get('totalPrice'),
                'qtd_parcelas': dado.get('numParcels'),
                'valor_parcela': dado.get('parcelValue'),
                'total_parcelado': dado.get('totalParceledValue'),
                'tem_juros': dado.get("hasInterest"),
                'marketplace': dado.get('sellerMarketplace'),
                'estoque': dado.get('stock'),
                'indice_competitividade': dado.get('competitivenessIndex'),
                'desconto_perc': dado.get('lowerInPercentage'),
                'url_imagem': dado.get('imageUrl'),
            }

            variacoes = dado.get('skuVariations', [])
            if isinstance(variacoes, list):
                for var in variacoes:
                    if isinstance(var, dict) and 'selector_name' in var and 'value_name' in var:
                        oferta[var['selector_name']] = var['value_name']

            lista_ofertas.append(oferta)

        for valor in dado.values():
            buscar_ofertas_completas(valor, lista_ofertas)

    elif isinstance(dado, list):
        for item in dado:
            buscar_ofertas_completas(item, lista_ofertas)
print('Extração feita com sucesso!')

Extração feita com sucesso!


Coleta dos links dos produtos, categorias Smart TV e Notebook Gamer 

In [3]:
categorias = {
    'Smart TV': 'https://www.buscape.com.br/search?q=smart%20tv',
    'Notebook Gamer': 'https://www.buscape.com.br/search?q=notebook%20gamer'
}

produtos_links = []

print("Iniciando a coleta dos links dos produtos...\n")

for nome_categoria, url_busca in categorias.items():
    try:
        response = requests.get(url_busca, headers=headers)
        if response.status_code == 200:
            soup = BeautifulSoup(response.text, 'html.parser')
            
            tags_link = soup.find_all('a', href=True) # find_all('a') pega TODOS os links da página (menu, rodapé, busca relacionada...)
            links_categoria = 0             # por isso depois a gente filtra, senão entra carregador, película, celular etc.

            for tag in tags_link:
                href = tag['href']
                href_lower = href.lower()  # minúsculo para comparar sem se preocupar com maiúscula

                # Esses trechos NÃO são página de produto. Ex.: /busca/s25 era busca de celular
                # e acabava no DataFrame como se fosse Smart TV.
                # continue = "pula este link e vai para o próximo".
                ignorar = ['/busca/', '/search', 'javascript:', '#']
                if any(parte in href_lower for parte in ignorar):
                    continue

                # Só aceita link que parece da categoria que estamos coletando.
                # Se estou em Smart TV e o href não tem "tv", descarta.
                if nome_categoria == 'Smart TV' and 'tv' not in href_lower:
                    continue
                if nome_categoria == 'Notebook Gamer' and 'notebook' not in href_lower:
                    continue

                # Link relativo (/tv/samsung-...) vira URL completa do Buscapé.
                url_completa = href if href.startswith('http') else f"https://www.buscape.com.br{href}"

                if not any(item['url'] == url_completa for item in produtos_links):
                    produtos_links.append({
                        'categoria_alvo': nome_categoria,
                        'url': url_completa
                    })
                    links_categoria += 1

            print(f"Encontrados {links_categoria} produtos para {nome_categoria}")
        else:
            print("Erro ao acessar os dados")
    except Exception as e:
        print(f"erro durante a requisição: {e}")

    time.sleep(1)

print(f"Busca concluída! Total de produtos únicos armazenados: {len(produtos_links)}")

Iniciando a coleta dos links dos produtos...

Encontrados 43 produtos para Smart TV
Encontrados 42 produtos para Notebook Gamer
Busca concluída! Total de produtos únicos armazenados: 85


Raspagem e DataFrame

In [4]:
todas_as_ofertas = []

for idx, item_produto in enumerate(produtos_links, start=1):
    url_produto = item_produto['url']
    categoria_produto = item_produto['categoria_alvo']

    try:
        res_produto = requests.get(url_produto, headers=headers)
        if res_produto.status_code == 200:
            soup_prod = BeautifulSoup(res_produto.text, 'html.parser')
            script_next = soup_prod.find('script', id='__NEXT_DATA__')

            if script_next and script_next.text:
                dados_json = json.loads(script_next.text)

                ofertas_da_pagina = []
                buscar_ofertas_completas(dados_json, ofertas_da_pagina)
                data_hora = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

                for oferta in ofertas_da_pagina:
                    oferta['categoria'] = categoria_produto
                    oferta['link_produto'] = url_produto
                    oferta['data_hora_extracao'] = data_hora
                    todas_as_ofertas.append(oferta)

                print(f"Ofertas da página {idx} coletadas com sucesso!")
            else:
                print(f"JSON não encontrado na página {idx}: {url_produto}")
        else:
            print(f"Falha ao acessar a página. {res_produto.status_code}")
    except Exception as e:
        print(f"Erro ao acessar o link: {e}")

    time.sleep(1)

df_bruto = pd.DataFrame(todas_as_ofertas)

# Rede de segurança: mesmo com o filtro de URL, uma página de TV pode trazer acessório.
# Aqui a gente olha o NOME do anúncio e fica só com o que tem "smart tv" ou "notebook".
# r'smart\s*tv' aceita "Smart TV" e "SmartTV" (\s* = espaço opcional).
# case=False ignora maiúscula/minúscula. na=False trata nome vazio como False.
if not df_bruto.empty and 'nome_anuncio' in df_bruto.columns:
    df_bruto = df_bruto[
        df_bruto['nome_anuncio'].str.contains(r'smart\s*tv|notebook', case=False, na=False)
    ]

df_bruto.to_csv('ofertas_buscape_bruto.csv', index=False, encoding='utf-8')

Ofertas da página 1 coletadas com sucesso!
Ofertas da página 2 coletadas com sucesso!
Ofertas da página 3 coletadas com sucesso!
Ofertas da página 4 coletadas com sucesso!
Ofertas da página 5 coletadas com sucesso!
Ofertas da página 6 coletadas com sucesso!
Ofertas da página 7 coletadas com sucesso!
Ofertas da página 8 coletadas com sucesso!
Ofertas da página 9 coletadas com sucesso!
Ofertas da página 10 coletadas com sucesso!
Ofertas da página 11 coletadas com sucesso!
Ofertas da página 12 coletadas com sucesso!
Ofertas da página 13 coletadas com sucesso!
Ofertas da página 14 coletadas com sucesso!
Ofertas da página 15 coletadas com sucesso!
Ofertas da página 16 coletadas com sucesso!
Ofertas da página 17 coletadas com sucesso!
Ofertas da página 18 coletadas com sucesso!
Ofertas da página 19 coletadas com sucesso!
Ofertas da página 20 coletadas com sucesso!
Ofertas da página 21 coletadas com sucesso!
Ofertas da página 22 coletadas com sucesso!
Ofertas da página 23 coletadas com sucess

In [5]:
display(df_bruto)

,loja,preco_vista,nome_anuncio,id_oferta,preco_total,qtd_parcelas,valor_parcela,total_parcelado,tem_juros,marketplace,estoque,indice_competitividade,desconto_perc,url_imagem,categoria,link_produto,data_hora_extracao
3,Magazine Luiza,151.91,Smart Tv Box Streaming Android Tv 4k Sua Tv Em...,NaN,NaN,NaN,NaN,NaN,True,None,NaN,0.0000,None,NaN,Smart TV,https://www.buscape.com.br/smart-tv-box,2026-10-07 20:34:23
4,Magazine Luiza,449.26,Smart TV Box 4K Intelbras,NaN,NaN,NaN,NaN,NaN,True,None,NaN,0.0000,None,NaN,Smart TV,https://www.buscape.com.br/smart-tv-box,2026-10-07 20:34:23
7,Magazine Luiza,229.90,Aparelho Tv Stick Wi fi Smart Tv Android Hdmi ...,NaN,NaN,NaN,NaN,NaN,False,None,NaN,0.0000,None,NaN,Smart TV,https://www.buscape.com.br/smart-tv-box,2026-10-07 20:34:23
14,Amazon,269.55,Smart TV Box Tv Stick Aquário STV-3000 Plus – ...,NaN,NaN,NaN,NaN,NaN,True,None,NaN,0.0000,None,NaN,Smart TV,https://www.buscape.com.br/smart-tv-box,2026-10-07 20:34:23
17,Fast Shop,1499.90,Smart TV 43” Britânia LED Dolby Audio Roku TV ...,1544222095,1499.90,0.0,0.0,0.0,True,None,NaN,0.6877,None,https://i.zst.com.br/thumbs/49/34/3a/-16341752...,Smart TV,https://www.buscape.com.br/tv/smart-tv-43-brit...,2026-10-07 20:34:26
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
302,Amazon,11291.57,"Notebook ASUS ROG Strix G16, Intel Core i9, RT...",1534888714,11291.57,0.0,0.0,0.0,True,None,1.0,0.7090,None,https://i.zst.com.br/thumbs/49/e/1d/-168618989...,Notebook Gamer,https://www.buscape.com.br/notebook/notebook-g...,2026-10-07 20:36:20
303,KaBuM!,11899.15,Notebook Gamer Rog Strix G16 G615jmr Nvidia RT...,1501783535,13999.00,10.0,1399.9,13999.0,True,None,NaN,0.6001,None,https://i.zst.com.br/thumbs/49/2e/14/-16755454...,Notebook Gamer,https://www.buscape.com.br/notebook/notebook-g...,2026-10-07 20:36:20
304,Fast Shop,10659.18,Notebook Gamer ROG Strix G16 G615JMR NVIDIA RT...,1543926603,10659.18,0.0,0.0,0.0,True,None,NaN,0.8646,None,https://i.zst.com.br/thumbs/49/1e/35/-16336883...,Notebook Gamer,https://www.buscape.com.br/notebook/notebook-g...,2026-10-07 20:36:20
305,Amazon,11291.57,"Notebook ASUS ROG Strix G16, Intel Core i9, RT...",1534888714,11291.57,0.0,0.0,0.0,True,None,1.0,0.7090,None,https://i.zst.com.br/thumbs/49/e/1d/-168618989...,Notebook Gamer,https://www.buscape.com.br/notebook/notebook-g...,2026-10-07 20:36:20


In [8]:
import re
import numpy as np
import pandas as pd

df_limpo = df_bruto.copy()# Cópia para preservar os dados originais

# Remoção de duplicatas pelo id_oferta
if 'id_oferta' in df_limpo.columns:
    df_limpo = df_limpo.drop_duplicates(subset=['id_oferta'], keep='first')

# Percentual de dados faltantes por coluna 
print("Percentual de valores nulos (%):")
display((df_limpo.isna().mean() * 100).round(2).astype(str) + '%')

# Preenchimento de ausentes:
if 'marketplace' in df_limpo.columns:
    df_limpo['marketplace'] = df_limpo['marketplace'].fillna(False)

if 'qtd_parcelas' in df_limpo.columns:
    df_limpo['qtd_parcelas'] = df_limpo['qtd_parcelas'].fillna(1)

if 'desconto_perc' in df_limpo.columns:
    df_limpo['desconto_perc'] = df_limpo['desconto_perc'].fillna(0)  # 0 indica ausência de desconto

if 'preco_vista' in df_limpo.columns and 'preco_total' in df_limpo.columns:
    df_limpo['preco_vista'] = df_limpo['preco_vista'].fillna(df_limpo['preco_total'])

# Preços e valores monetários para float
colunas_float = ['preco_vista', 'preco_total', 'valor_parcela', 'total_parcelado']
for col in colunas_float:
    if col in df_limpo.columns:
        df_limpo[col] = pd.to_numeric(df_limpo[col], errors='coerce').astype(float)

# Quantidade de parcelas para int
if 'qtd_parcelas' in df_limpo.columns:
    df_limpo['qtd_parcelas'] = pd.to_numeric(df_limpo['qtd_parcelas'], errors='coerce').fillna(1).astype(int)


# Extração de polegadas
def extrair_polegadas(nome):
    if pd.isna(nome):
        return None
    match = re.search(r"(\d{2})\s*(?:\"|'|polegadas|polegada|pol\b)", str(nome), re.IGNORECASE)
    return int(match.group(1)) if match else None

# Extração de memória RAM
def extrair_ram(nome):
    if pd.isna(nome):
        return None
    match = re.search(r'(\d{1,2})\s*GB', str(nome), re.IGNORECASE)
    return int(match.group(1)) if match else None

df_limpo['polegadas'] = df_limpo.apply(
    lambda row: extrair_polegadas(row['nome_anuncio']) if row.get('categoria') == 'Smart TV' else None,
    axis=1,
)

df_limpo['ram_gb'] = df_limpo.apply(
    lambda row: extrair_ram(row['nome_anuncio']) if row.get('categoria') == 'Notebook Gamer' else None,
    axis=1,
)

#Custo Efetivo do Parcelamento (%)
df_limpo['acrescimo_parcelamento_pct'] = np.where(
    (df_limpo['preco_vista'] > 0) & (df_limpo['total_parcelado'].notnull()),
    ((df_limpo['total_parcelado'] - df_limpo['preco_vista']) / df_limpo['preco_vista']) * 100,
    0.0
)

df_limpo['acrescimo_parcelamento_pct'] = df_limpo['acrescimo_parcelamento_pct'].clip(lower=0).round(2)


display(df_limpo.head())

Percentual de valores nulos (%):


loja                        0.0%
preco_vista                 0.0%
nome_anuncio                0.0%
id_oferta                  0.81%
preco_total                0.81%
qtd_parcelas               0.81%
valor_parcela              0.81%
total_parcelado            0.81%
tem_juros                   0.0%
marketplace               100.0%
estoque                   31.71%
indice_competitividade      0.0%
desconto_perc             100.0%
url_imagem                 0.81%
categoria                   0.0%
link_produto                0.0%
data_hora_extracao          0.0%
dtype: str

,loja,preco_vista,nome_anuncio,id_oferta,preco_total,qtd_parcelas,valor_parcela,total_parcelado,tem_juros,marketplace,estoque,indice_competitividade,desconto_perc,url_imagem,categoria,link_produto,data_hora_extracao,polegadas,ram_gb,acrescimo_parcelamento_pct
3,Magazine Luiza,151.91,Smart Tv Box Streaming Android Tv 4k Sua Tv Em...,NaN,NaN,1,NaN,NaN,True,False,NaN,0.0000,0,NaN,Smart TV,https://www.buscape.com.br/smart-tv-box,2026-10-07 20:34:23,NaN,NaN,0.0
17,Fast Shop,1499.90,Smart TV 43” Britânia LED Dolby Audio Roku TV ...,1544222095,1499.90,0,0.00,0.00,True,False,NaN,0.6877,0,https://i.zst.com.br/thumbs/49/34/3a/-16341752...,Smart TV,https://www.buscape.com.br/tv/smart-tv-43-brit...,2026-10-07 20:34:26,NaN,NaN,0.0
18,Webcontinental,1649.90,"Smart TV Britânia BTV43VA4REGB 43"" Full HD LED...",1501991139,1649.84,8,206.23,1649.84,False,False,NaN,0.4910,0,https://image.zst.com.br/A9EF4B87E47D5B3B9991F...,Smart TV,https://www.buscape.com.br/tv/smart-tv-43-brit...,2026-10-07 20:34:26,43.0,NaN,0.0
20,Amazon,1566.80,Smart TV 43” Britânia BTV43VA4REGB Roku TV LED...,1506804496,1566.80,0,0.00,0.00,True,False,2.0,0.5925,0,https://i.zst.com.br/thumbs/49/2b/3d/-15571756...,Smart TV,https://www.buscape.com.br/tv/smart-tv-43-brit...,2026-10-07 20:34:26,NaN,NaN,0.0
25,Fast Shop,2138.05,"Smart TV 50"" Britânia Ultra HD LED BTV50VA4REG...",1543953536,2138.05,0,0.00,0.00,True,False,NaN,0.6916,0,https://i.zst.com.br/thumbs/49/2f/14/-16753202...,Smart TV,https://www.buscape.com.br/tv/smart-tv-50-brit...,2026-10-07 20:34:28,50.0,NaN,0.0


In [9]:
#Análise

for cat in df_limpo['categoria'].unique():
    print(f"\n--- Categoria: {cat} ---")
    df_cat = df_limpo[df_limpo['categoria'] == cat]
    
    # Top 5 lojas em volume
    top5 = df_cat['loja'].value_counts().head(5).index
    df_top5 = df_cat[df_cat['loja'].isin(top5)]
    
    # Preço médio de cada uma das 5 lojas
    resumo = df_top5.groupby('loja')['preco_vista'].agg(['count', 'mean']).reindex(top5)
    resumo.columns = ['Qtd Ofertas', 'Preço Médio (R$)']
    
    print(resumo.round(2))

comp_mkt = df_limpo.groupby(['categoria', 'marketplace'])['preco_vista'].agg(['count', 'mean']).round(2)
comp_mkt.columns = ['Qtd Ofertas', 'Preço Médio (R$)']
print(comp_mkt)


--- Categoria: Smart TV ---
                Qtd Ofertas  Preço Médio (R$)
loja                                         
Magazine Luiza           58           2156.34
Amazon                   15           2025.71
Fast Shop                13           2479.56
KaBuM!                    5           2886.82
Webcontinental            4           2886.96

--- Categoria: Notebook Gamer ---
                Qtd Ofertas  Preço Médio (R$)
loja                                         
Amazon                    9           7817.17
KaBuM!                    8           8324.45
Magazine Luiza            5           8002.68
Fast Shop                 3           8456.83
                            Qtd Ofertas  Preço Médio (R$)
categoria      marketplace                               
Notebook Gamer False                 25           8093.36
Smart TV       False                 98           2241.10


## Relatório de Análise de Mercado: Smart TVs e Notebooks Gamers

Nesta etapa, foi o cruzamento dos dados tratados para responder a duas perguntas fundamentais sobre o comportamento das principais lojas e o modelo de vendas (Marketplace vs. Loja Própria).

---

### 1. Top 5 Lojas por Volume e Preço Médio

Analisando o volume de ofertas cadastradas e o preço médio praticado por cada varejista em cada categoria, o resultado foi o seguinte:


* **Smart TVs:** A **Magazine Luiza** lidera em volume de ofertas cadastradas (58 no total), mantendo um preço médio competitivo na faixa de R$ 2.156,00. Já plataformas como KaBuM! e Webcontinental apresentam preços médios mais elevados, próximos a R$ 2.887,00.


* **Notebook Gamer:** A **Amazon** destaca-se no volume de ofertas (9 no total) com um preço médio de R$ 7.817,17, enquanto a **Fast Shop** registra a média de preço mais alta da categoria R$ 8.456,83.

---

### 2. Comparativo: Marketplace vs. Loja Própria

Já a proporção e a média de preços entre produtos vendidos diretamente pela loja ou através de intermediários (Marketplace):

#### Principais Insights do Modelo de Vendas:
* **Exclusividade de Vendas Diretas:** Todos os registros capturados e analisados para ambas as categorias (`Smart TV` e `Notebook Gamer`) retornaram `False` para a coluna `marketplace`, indicando que a amostra atual é composta exclusivamente por **vendas diretas (loja própria)**.

* Demonstrando que as ofertas coletadas vieram diretamente das grandes marcas, garantindo maior uniformidade no preço à vista para as categorias.
